# Lab dl_01 · Build micrograd: scalar autograd and a tiny MLP

**Goal.** Write a ~150-line automatic-differentiation engine. Every number is a `Value` that remembers the operation and inputs
that produced it. Calling `loss.backward()` walks the graph in reverse topological order and applies the chain rule.

### Theory recap
For a node $o = f(a, b)$ with upstream gradient $\bar o = \partial L/\partial o$, the chain rule gives
$$\bar a \mathrel{+}= \frac{\partial f}{\partial a}\,\bar o, \qquad \bar b \mathrel{+}= \frac{\partial f}{\partial b}\,\bar o.$$
The **`+=`** matters: if a value feeds several nodes, the gradients from all paths **add** (multivariate chain rule).
Local derivatives you need:

| op | forward | local derivative |
|---|---|---|
| $a+b$ | | $1, 1$ |
| $a\cdot b$ | | $b, a$ |
| $a^k$ | | $k a^{k-1}$ |
| $\tanh a$ | $t$ | $1-t^2$ |
| $\mathrm{ReLU}(a)$ | | $\mathbb 1[a>0]$ |
| $e^a$ | | $e^a$ |
| $\log a$ | | $1/a$ |

Backprop order: build a topological ordering of the graph (children before parents), set $\bar L = 1$, then call each node's
`_backward` in **reverse** order so a node's gradient is complete before it is propagated further.

**How this works.** Fill in every cell marked `# TODO` (they currently `raise NotImplementedError`). Run the notebook top to bottom with the working directory set to `labs/`. The last cell runs hidden checks on your functions and records the result in the Learning Center. Do the paper homework for the unlocking skill first — the lab assumes you can do the math by hand.

*This lab is original course material, inspired by Andrej Karpathy's open-source teaching code ([micrograd](https://github.com/karpathy/micrograd), [nn-zero-to-hero](https://github.com/karpathy/nn-zero-to-hero), [nanoGPT](https://github.com/karpathy/nanoGPT), [build-nanogpt](https://github.com/karpathy/build-nanogpt), [minbpe](https://github.com/karpathy/minbpe)). Watching the corresponding lecture after finishing is highly recommended.*

In [ ]:
import math
import random
import matplotlib.pyplot as plt

## 1. The `Value` class
Implement the TODO methods. Each op creates `out = Value(result, (inputs...), op_name)` and attaches an `out._backward`
closure that **accumulates** into the inputs' `.grad`. Subtraction, division and negation can be built from `+`, `*` and `**`.

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op=""):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        # TODO: forward value and _backward (d(ab)/da = b, d(ab)/db = a)
        raise NotImplementedError

    def __pow__(self, k):
        assert isinstance(k, (int, float)), "only constant exponents"
        # TODO
        raise NotImplementedError

    def __neg__(self):          # -self
        return self * -1

    def __sub__(self, other):   # self - other
        return self + (-other)

    def __truediv__(self, other):  # self / other  (hint: multiply by other ** -1)
        # TODO
        raise NotImplementedError

    def __radd__(self, other):  # other + self
        return self + other

    def __rsub__(self, other):  # other - self
        return Value(other) - self

    def __rmul__(self, other):  # other * self
        return self * other

    def __rtruediv__(self, other):  # other / self
        return Value(other) / self

    def tanh(self):
        # TODO
        raise NotImplementedError

    def relu(self):
        # TODO
        raise NotImplementedError

    def exp(self):
        # TODO
        raise NotImplementedError

    def log(self):
        # TODO
        raise NotImplementedError

    def backward(self):
        # TODO: topological sort of the graph rooted at self, set self.grad = 1, call _backward in reverse order
        raise NotImplementedError

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

### Sanity check by hand
Before running: on paper compute all gradients of $f = (a b + c)\,d$ at $a=3, b=-2, c=5, d=4$. Then compare.

In [ ]:
a, b, c, d = Value(3.0), Value(-2.0), Value(5.0), Value(4.0)
f = (a * b + c) * d
f.backward()
print(f, a.grad, b.grad, c.grad, d.grad)

# numerical gradient check (central differences) for a messier expression
def expr(x, y):
    return ((x * y).tanh() + (x ** 3).relu() + (y / 2).exp() - (x - y) ** 2)

x, y = Value(0.7), Value(-1.3)
out = expr(x, y); out.backward()
h = 1e-6
num_dx = (expr(Value(0.7 + h), Value(-1.3)).data - expr(Value(0.7 - h), Value(-1.3)).data) / (2 * h)
print("analytic dx =", x.grad, " numeric dx =", num_dx)

## 2. Neurons, layers, MLPs
A neuron computes $\tanh(\mathbf w\cdot\mathbf x + b)$ (or the plain affine value if `nonlin=False`). Initialise
$w_i \sim U(-1, 1)$ and $b = 0$. **Parameter order matters for the checks:** `Neuron.parameters()` returns `self.w + [self.b]`,
`Layer` concatenates its neurons, `MLP` concatenates its layers. In `MLP(nin, nouts)` every layer uses tanh **except the last**,
which is linear. A layer with a single neuron returns that `Value` directly instead of a list.

In [ ]:
class Neuron:
    def __init__(self, nin, nonlin=True):
        # TODO: self.w (list of Values), self.b (Value), self.nonlin
        raise NotImplementedError

    def __call__(self, x):
        # TODO
        raise NotImplementedError

    def parameters(self):
        return self.w + [self.b]


class Layer:
    def __init__(self, nin, nout, nonlin=True):
        # TODO: self.neurons
        raise NotImplementedError

    def __call__(self, x):
        # TODO (return a single Value if nout == 1)
        raise NotImplementedError

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]


class MLP:
    def __init__(self, nin, nouts):
        # TODO: self.layers; last layer nonlin=False
        raise NotImplementedError

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

## 3. Training loop
`train(model, xs, ys, steps, lr)` runs full-batch gradient descent on the mean squared error
$L = \frac1n\sum_i (\hat y_i - y_i)^2$ and returns the list of losses (floats). Remember to **zero the gradients** every step
(why? see the `+=` above).

In [ ]:
def zero_grad(params):
    for p in params:
        p.grad = 0.0


def mse_loss(preds, targets):
    # TODO: mean squared error as a Value
    raise NotImplementedError


def train(model, xs, ys, steps=50, lr=0.05):
    # TODO: forward, loss, zero_grad, backward, SGD update p.data -= lr * p.grad; record loss.data
    raise NotImplementedError

## 4. Experiment: learn XOR
XOR is not linearly separable — a hidden layer is required.

In [ ]:
random.seed(0)
model = MLP(2, [8, 1])
xs = [[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]]
ys = [-1.0, 1.0, 1.0, -1.0]
losses = train(model, xs, ys, steps=200, lr=0.1)
plt.plot(losses); plt.yscale("log"); plt.xlabel("step"); plt.ylabel("MSE"); plt.title("XOR"); plt.show()
print([round(model(x).data, 3) for x in xs])

### Reflect (on paper)
1. Why does the gradient of a value used twice need `+=`? Construct a 2-node example where `=` gives the wrong answer.
2. Remove the hidden layer (`MLP(2, [1])`). What is the best loss you can reach on XOR, and why?
3. How many `Value` objects does one forward pass of `MLP(2, [8, 1])` on one input create? Why is PyTorch's tensor-level graph so much faster?

## Grade
Runs the hidden checks (fast, small inputs, fixed seeds) and reports to the local app.

In [ ]:
from lc_client import grade
grade("dl_01_micrograd", globals())